# 10-714 Homework 4

In this homework, you will leverage all of the components built in the last three homeworks to solve some modern problems with high performing network structures. We will start by adding a few new ops leveraging our new CPU/CUDA backends. Then, you will implement convolution, and a convolutional neural network to train a classifier on the CIFAR-10 image classification dataset. Then, you will implement recurrent and long-short term memory (LSTM) neural networks, and do word-level prediction language modeling on the Penn Treebank dataset.

As always, we will start by copying this notebook and getting the starting code.
Reminder: __you must save a copy in drive__.

In [1]:
# Mount Drive (same as before)
from google.colab import drive
drive.mount('/content/drive')

# %cd /content/drive/MyDrive/10714

# Clone YOUR repo into a folder named "hw4"
# !git clone https://github.com/3N3G/dlsys-final.git proj
%cd /content/drive/MyDrive/10714/proj

# Same installs as before
!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/10714/proj
  Cloning https://github.com/dlsys10714/mugrade.git to /tmp/pip-req-build-b7adr7wh
  Running command git clone --filter=blob:none --quiet https://github.com/dlsys10714/mugrade.git /tmp/pip-req-build-b7adr7wh
  Resolved https://github.com/dlsys10714/mugrade.git to commit ac73f725eb2ce0e2c6a38fa540035ee970b8b873
  Preparing metadata (setup.py) ... done


In [ ]:
!make

CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- Found pybind11: /usr/local/lib/python3.12/dist-packages/pybind11/include (found version "3.0.1")
-- Found cuda, building cuda backend
-- Configuring done (0.7s)
-- Generating done (0.5s)
-- Build files have been written to: /content/drive/MyDrive/10714/proj/build
make[1]: Entering directory '/content/drive/MyDrive/10714/proj/build'
make[2]: Entering directory '/content/drive/MyDrive/10714/proj/build'
make[3]: Entering directory '/content/drive/MyDrive/10714/proj/build'
make[3]: Leaving directory '/content/drive/MyDrive/10714/proj/build'
[  0%] Built target ndarray_backend_cpu
make[3]: Entering directory '/content/drive/

In [2]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

env: PYTHONPATH=./python
env: NEEDLE_BACKEND=nd


In [3]:
import sys
sys.path.append('./python')

In [4]:
# Download the datasets you will be using for this assignment

import urllib.request
import os

!mkdir -p './data/ptb'
# Download Penn Treebank dataset
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

# Download CIFAR-10 dataset
if not os.path.isdir("./data/cifar-10-batches-py"):
    urllib.request.urlretrieve("https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", "./data/cifar-10-python.tar.gz")
    !tar -xvzf './data/cifar-10-python.tar.gz' -C './data'

You will now use your convolutional layer to implement a model similar to _ResNet9_, which is known to be a reasonable model for getting good accuracy on CIFAR-10 quickly (see [here](https://github.com/davidcpage/cifar10-fast)). Our main change is that we used striding instead of pooling and divided all of the channels by 4 for the sake of performance (as our framework is not as well-optimized as industry-grade frameworks).

In the figure below, before the first linear layer, you should "flatten" the tensor. You can use the module `Flatten` in `nn_basic.py`, or you can simply use `.reshape` in the `forward()` method of your ResNet9.

Make sure that you pass the device to all modules in your model; otherwise, you will get errors about mismatched devices when trying to run with CUDA.

<center><img src="https://github.com/dlsyscourse/hw4/blob/main/ResNet9.png?raw=true" alt="ResNet9" style="width: 400px;" /></center>

We have tried to make it easier to pass the tests here than for previous assignments where you have implemented models. In particular, we are just going to make sure it has the right number of parameters and similar accuracy and loss after 1 or 2 batches of CIFAR-10.

Now, you can train your model on CIFAR-10 using the following code. Note that this is likely going to be quite slow, and also  not all that accurate due to the lack of data augmentation. You should expect it to take around 500s per epoch.

In [5]:
import sys
sys.path.append('./python')
sys.path.append('./apps')

import importlib
import needle
importlib.reload(needle)
import models
importlib.reload(models)
import simple_ml
importlib.reload(simple_ml)

import needle as ndl
from models import ResNet9
from simple_ml import train_cifar10, evaluate_cifar10

print("HERE")
device = ndl.cuda()
dataset = ndl.data.CIFAR10Dataset("data/cifar-10-batches-py", train=True)
print("LOADING DATA")
dataloader = ndl.data.DataLoader(\
         dataset=dataset,
         batch_size=128,
         shuffle=True,)
print("MODEL")
model = ResNet9(device=device, dtype="float32")
print("TRAINING")
train_cifar10(model, dataloader, n_epochs=30, optimizer=ndl.optim.Adam,
      lr=0.001, weight_decay=0.001)
evaluate_cifar10(model, dataloader)

HERE
LOADING DATA
MODEL
TRAINING


  3%|▎         | 1/30 [00:52<25:19, 52.38s/it]

ACCURACY  0.38966 LOSS  1.6994773844146729


  7%|▋         | 2/30 [01:42<23:57, 51.34s/it]

ACCURACY  0.49442 LOSS  1.4006506411361694


...

 97%|█████████▋| 29/30 [25:03<00:51, 51.66s/it]

ACCURACY  0.83648 LOSS  0.4514553662490845


100%|██████████| 30/30 [25:55<00:00, 51.86s/it]

ACCURACY  0.8409 LOSS  0.4367025932312012


(0.75746, 0.7388674331569671)

In [1]:
# ================================
# 0. Setup & imports
# ================================
!pip install -q torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# 1. CIFAR-10 data loaders
# ================================
transform = T.Compose([
    T.ToTensor(),                     # [0, 1]
    T.Normalize((0.5, 0.5, 0.5),      # mean
                (0.5, 0.5, 0.5)),     # std
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=2)

# ================================
# 2. ResNet9 definition (PyTorch)
#    matches your Needle version:
#    - ConvBN(3,16,7,4)
#    - ConvBN(16,32,3,2)
#    - ConvBN(32,32,3,1) x2 + residual
#    - ConvBN(32,64,3,2)
#    - ConvBN(64,128,3,2)
#    - ConvBN(128,128,3,1) x2 + residual
#    - Flatten, Linear(128,128), ReLU, Linear(128,10)
# ================================
class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k, s):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=k, stride=s,
                      padding=k // 2, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResNet9(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Layer 1-8 conv blocks
        self.conv1 = ConvBNReLU(3,   16, 7, 4)
        self.conv2 = ConvBNReLU(16,  32, 3, 2)
        self.conv3 = ConvBNReLU(32,  32, 3, 1)
        self.conv4 = ConvBNReLU(32,  32, 3, 1)

        self.conv5 = ConvBNReLU(32,  64, 3, 2)
        self.conv6 = ConvBNReLU(64, 128, 3, 2)
        self.conv7 = ConvBNReLU(128,128, 3, 1)
        self.conv8 = ConvBNReLU(128,128, 3, 1)

        # Fully connected part
        # After conv6 on CIFAR-10 with these strides, spatial size is 1x1,
        # so feature dim = 128.
        self.fc1 = nn.Linear(128, 128)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Input: (N, 3, 32, 32)
        x = self.conv1(x)
        out2 = self.conv2(x)

        x = self.conv3(out2)
        x = self.conv4(x)

        # Residual from layer 2
        x = x + out2

        x = self.conv5(x)
        out6 = self.conv6(x)

        x = self.conv7(out6)
        x = self.conv8(x)

        # Flatten both x and out6, add residual
        x_flat   = x.view(x.size(0), -1)      # (N, 128 * 1 * 1) = (N, 128)
        out6_flat = out6.view(out6.size(0), -1)
        x = x_flat + out6_flat

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


model = ResNet9().to(device)
print(model)

# ================================
# 3. Loss & optimizer
# ================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

# ================================
# 4. Train / eval helpers
# ================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


# ================================
# 5. Main training loop
# ================================
num_epochs = 10   # set to 20/30 to mirror your Needle run

for epoch in range(1, num_epochs + 1):
    train_acc, train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_acc, test_loss   = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f} | "
        f"test_acc={test_acc:.4f}, test_loss={test_loss:.4f}"
    )


Using device: cuda


100%|██████████| 170M/170M [00:04<00:00, 37.2MB/s]


ResNet9(
  (conv1): ConvBNReLU(
    (block): Sequential(
      (0): Conv2d(3, 16, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
  )
  (conv2): ConvBNReLU(
    (block): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
  )
  (conv3): ConvBNReLU(
    (block): Sequential(
      (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
  )
  (conv4): ConvBNReLU(
    (block): Sequential(
      (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1,

In [2]:
# ==== 0. (Optional) install, usually already in Colab ====
# !pip install -q torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==== 1. Data: CIFAR-10 loaders ====
transform = T.Compose([
    T.ToTensor(),                     # [0,1]
    T.Normalize((0.5, 0.5, 0.5),      # mean
                (0.5, 0.5, 0.5)),     # std
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=2)

# ==== 2. Model: simple ResNet (torchvision) ====
from torchvision.models import resnet18

model = resnet18(weights=None, num_classes=10)  # plain ResNet-18 for CIFAR-10
model = model.to(device)

# ==== 3. Loss & optimizer ====
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

# ==== 4. Train & eval loops ====
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total, running_loss / total


# ==== 5. Main training loop ====
num_epochs = 10   # change to 20 / 30 to match your needle run

for epoch in range(1, num_epochs + 1):
    train_acc, train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_acc, test_loss   = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_acc={train_acc:.4f}, train_loss={train_loss:.4f} | "
        f"test_acc={test_acc:.4f}, test_loss={test_loss:.4f}"
    )


Using device: cuda
Epoch 01 | train_acc=0.5087, train_loss=1.3662 | test_acc=0.5821, test_loss=1.1729
Epoch 02 | train_acc=0.6514, train_loss=0.9894 | test_acc=0.6646, test_loss=0.9621
Epoch 03 | train_acc=0.7086, train_loss=0.8377 | test_acc=0.7112, test_loss=0.8284
Epoch 04 | train_acc=0.7382, train_loss=0.7504 | test_acc=0.7190, test_loss=0.8205
Epoch 05 | train_acc=0.7654, train_loss=0.6812 | test_acc=0.7139, test_loss=0.8404
Epoch 06 | train_acc=0.7833, train_loss=0.6286 | test_acc=0.7334, test_loss=0.7939
Epoch 07 | train_acc=0.8010, train_loss=0.5763 | test_acc=0.7091, test_loss=0.8610
Epoch 08 | train_acc=0.8211, train_loss=0.5229 | test_acc=0.7240, test_loss=0.8168
Epoch 09 | train_acc=0.8375, train_loss=0.4771 | test_acc=0.7464, test_loss=0.7831
Epoch 10 | train_acc=0.8511, train_loss=0.4353 | test_acc=0.7515, test_loss=0.7546
